<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_W5_J4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prédiction du cours des actions avec LSTM et PyTorch
Ce notebook contient la solution complète pour le défi de prédiction boursière.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

# Configuration du device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Utilisation de : {device}')

## 2. Chargement et Prétraitement
Nous utilisons le fichier `/content/symbols_valid_meta.csv`. Note : Pour une prédiction temporelle réelle, nous simulerons une colonne de prix si les données sont purement des métadonnées.

In [ ]:
df = pd.read_csv('/content/symbols_valid_meta.csv')

# Simulation d'une série temporelle pour l'exercice si le CSV ne contient que des métadonnées
# (Le fichier fourni semble être une liste de symboles, nous créons une séquence fictive pour démontrer le LSTM)
if 'Close' not in df.columns:
    print("Simulation de données de clôture pour la démonstration...")
    np.random.seed(42)
    df['Close'] = np.cumsum(np.random.randn(len(df))) + 100

scaler = MinMaxScaler(feature_range=(-1, 1))
data_normalized = scaler.fit_transform(df['Close'].values.reshape(-1, 1))

def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

SEQ_LENGTH = 10
X, y = create_sequences(data_normalized, SEQ_LENGTH)

# Split train/test
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

## 3. Classe Dataset et DataLoader

In [ ]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=16, shuffle=True)
test_loader = DataLoader(StockDataset(X_test, y_test), batch_size=1, shuffle=False)

## 4. Définition du modèle LSTM

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_layer_size=50, output_size=1):
        super().__init__()
        self.hidden_layer_size = hidden_layer_size
        self.lstm = nn.LSTM(input_size, hidden_layer_size, batch_first=True)
        self.linear = nn.Linear(hidden_layer_size, output_size)

    def forward(self, input_seq):
        lstm_out, _ = self.lstm(input_seq)
        predictions = self.linear(lstm_out[:, -1, :])
        return predictions

model = LSTMModel().to(device)
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## 5. Entraînement

In [ ]:
epochs = 20
model.train()
for i in range(epochs):
    for seq, labels in train_loader:
        seq, labels = seq.to(device), labels.to(device)
        optimizer.zero_grad()
        y_pred = model(seq)
        single_loss = loss_function(y_pred, labels)
        single_loss.backward()
        optimizer.step()
    if i % 5 == 0:
        print(f'Epoch {i} perte: {single_loss.item():10.8f}')

## 6. Évaluation
Calcul du score R² et visualisation.

In [ ]:
model.eval()
preds = []
true_values = []
with torch.no_grad():
    for seq, labels in test_loader:
        seq = seq.to(device)
        preds.append(model(seq).item())
        true_values.append(labels.item())

# Inverser la normalisation
preds_rescaled = scaler.inverse_transform(np.array(preds).reshape(-1, 1))
true_rescaled = scaler.inverse_transform(np.array(true_values).reshape(-1, 1))

print(f"Score R² : {r2_score(true_rescaled, preds_rescaled):.4f}")

plt.figure(figsize=(12,6))
plt.plot(true_rescaled, label='Réel')
plt.plot(preds_rescaled, label='Prédit')
plt.legend()
plt.title('Prédiction du cours des actions')
plt.show()